In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2-0.5B-Instruct"

print(f"Loading model: {model_id}")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto"
    )

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=128,
        temperature=0.7,
        do_sample=True
    )

    llm = HuggingFacePipeline(pipeline=pipe)
    print("Model loaded successfully")
except Exception as e:
    print(f"Model load failed: {e}")
    print("Falling back to a simple local response generator.")
    tokenizer = None
    llm = None


def create_qwen_prompt(user_input: str, history: list | None = None):
    if history is None:
        history = []

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    messages.extend(history)
    messages.append(
        {
            "role": "user",
            "content": user_input
        }
    )

    if tokenizer is None:
        return user_input

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


print("Setup Complete")

e:\University\Flask\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model: Qwen/Qwen2-0.5B-Instruct


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1504.67it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded successfully
Setup Complete


import the libraries for flask app

In [4]:
import threading
import time
import requests
from flask import Flask, request, jsonify
from flask_cors import CORS
import logging

In [5]:
if 'llm' not in globals() or 'tokenizer' not in globals():
    raise EnvironmentError("The 'llm' variable is not defined. Please ensure that the model is loaded before starting the server.")
print("Enviroment check passed LLM found")

Enviroment check passed LLM found


In [6]:
def create_qwen_prompt(user_text, history=None):
    if history is None:
        history = []

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    messages.extend(history)
    messages.append(
        {
            "role": "user",
            "content": user_text
        }
    )

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def clean_qwen_response(response: str,prompt: str) -> str:
    if prompt in response:
        return response[len(prompt):].strip()
    if "<|im_start|>assistant" in response:
        return response.split("<|im_start|>assistant")[1].strip()
    return response.strip()

In [7]:
app = Flask(__name__)
CORS(app)
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)  # Set the logging level to ERROR to suppress info logs



In [8]:
@app.route('/chat', methods=['POST'])
def chat():
    try:
        data = request.get_json(silent=True) or {}
        user_message = data.get('message', '')
        formatted_prompt = create_qwen_prompt(user_message)

        if llm is None:
            return jsonify({
                "status": "success",
                "response": f"Local fallback response to: {user_message}"
            })

        raw_response = llm.invoke(formatted_prompt)
        ai_response = clean_qwen_response(raw_response, formatted_prompt)
        return jsonify({"status": "success", "response": ai_response})
    except Exception as e:
        return jsonify({"error": str(e)}), 500


In [ ]:
def run_flask_app():
    print("Starting Flask server on http://127.0.0.1:5000")
    app.run(host="127.0.0.1", port=5000, use_reloader=False)


run_flask_app()

Starting Flask server on http://127.0.0.1:5000
 * Serving Flask app '__main__'
 * Debug mode: off


In [ ]:
print("\n [TEST] Endpoint: /chat")
print("-" * 40)
try:
    resp = requests.post("http://127.0.0.1:5000/chat", json={"message": "Hello, how are you?"})
    if resp.status_code == 200:
        print(f"Response: {resp.json()}")
    else:
        print(f"Error: {resp.status_code}")
except requests.exceptions.RequestException as e:
    print(f"Request failed: {e}")
